# Fold NLP Model v3 — Multi-Head DistilBERT

Trains on `eda_dataset_v3.csv` with three classification heads:
- **Category** (10 classes): food, shopping, travel, etc.
- **Payment Method** (4 classes): cash, upi, card, unknown
- **Bank Account** (16+ classes): hdfc, sbi, slice, etc. + unknown

The model shares a DistilBERT encoder and has separate linear heads.
Combined loss = category_loss + 0.5 * method_loss + 0.5 * bank_loss

In [1]:
!pip install transformers datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from datasets import Dataset
import json

In [3]:
df = pd.read_csv("eda_dataset_v3.csv")
print(f"Loaded {len(df)} rows")
print(f"Columns: {list(df.columns)}")
print(f"\nCategory distribution:")
print(df['category'].value_counts())
print(f"\nPayment method distribution:")
print(df['payment_method'].value_counts())
print(f"\nText source distribution:")
print(df['text_source'].value_counts())

Loaded 42500 rows
Columns: ['text', 'amount', 'category', 'payment_method', 'payment_provider', 'bank_account', 'text_source']

Category distribution:
category
friends          5991
food             4400
healthcare       4372
shopping         4314
entertainment    4232
utilities        4208
investment       3809
emi              3744
education        3735
travel           3695
Name: count, dtype: int64

Payment method distribution:
payment_method
upi     23859
card    13045
cash     5596
Name: count, dtype: int64

Text source distribution:
text_source
hinglish_natural     13000
english_natural       9000
voice_transcript      5000
ocr_upi               4500
cue_row               3500
ocr_receipt           3000
friends_synthetic     2500
source_hinglish       1000
source_english        1000
Name: count, dtype: int64


In [4]:
# Clean data
df = df.dropna(subset=['text', 'category'])
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'].str.len() > 0]

VALID_CATEGORIES = {
    'education', 'emi', 'entertainment', 'food', 'friends',
    'healthcare', 'investment', 'shopping', 'travel', 'utilities'
}
df['category'] = df['category'].astype(str).str.strip().str.lower()
df = df[df['category'].isin(VALID_CATEGORIES)].copy()

# Normalize payment_method
VALID_METHODS = {'cash', 'upi', 'card'}
df['payment_method'] = df['payment_method'].fillna('unknown').astype(str).str.strip().str.lower()
df.loc[~df['payment_method'].isin(VALID_METHODS), 'payment_method'] = 'unknown'

# Normalize bank_account
df['bank_account'] = df['bank_account'].fillna('unknown').astype(str).str.strip().str.lower()
df.loc[df['bank_account'] == '', 'bank_account'] = 'unknown'

print(f"Rows after cleaning: {len(df)}")
print(f"\nPayment methods: {sorted(df['payment_method'].unique())}")
print(f"\nBank accounts (top 20): {df['bank_account'].value_counts().head(20).to_dict()}")

Rows after cleaning: 42500

Payment methods: ['card', 'cash', 'upi']

Bank accounts (top 20): {'unknown': 24308, 'hdfc': 3697, 'sbi': 3054, 'icici': 2177, 'axis': 1525, 'kotak': 1142, 'slice': 840, 'jupiter': 820, 'fi': 811, 'niyo': 775, 'pnb': 773, 'idfc': 471, 'yes bank': 452, 'bob': 444, 'indusind': 298, 'canara': 277, 'federal bank': 170, 'union bank': 164, 'bandhan': 157, 'rbl': 145}


In [5]:
# Encode labels (alphabetical order for reproducibility)
le_cat = LabelEncoder()
le_method = LabelEncoder()
le_bank = LabelEncoder()

df['label_cat'] = le_cat.fit_transform(df['category'])
df['label_method'] = le_method.fit_transform(df['payment_method'])
df['label_bank'] = le_bank.fit_transform(df['bank_account'])

num_cat = len(le_cat.classes_)
num_method = len(le_method.classes_)
num_bank = len(le_bank.classes_)

print(f"Category classes ({num_cat}): {list(le_cat.classes_)}")
print(f"Method classes ({num_method}): {list(le_method.classes_)}")
print(f"Bank classes ({num_bank}): {list(le_bank.classes_)}")

# Save label maps for inference
label_maps = {
    'category': {int(i): str(c) for i, c in enumerate(le_cat.classes_)},
    'payment_method': {int(i): str(c) for i, c in enumerate(le_method.classes_)},
    'bank_account': {int(i): str(c) for i, c in enumerate(le_bank.classes_)},
}
with open('label_maps_v3.json', 'w') as f:
    json.dump(label_maps, f, indent=2)
print("\nSaved label_maps_v3.json")

Category classes (10): ['education', 'emi', 'entertainment', 'food', 'friends', 'healthcare', 'investment', 'shopping', 'travel', 'utilities']
Method classes (3): ['card', 'cash', 'upi']
Bank classes (20): ['axis', 'bandhan', 'bob', 'canara', 'federal bank', 'fi', 'hdfc', 'icici', 'idfc', 'indusind', 'jupiter', 'kotak', 'niyo', 'pnb', 'rbl', 'sbi', 'slice', 'union bank', 'unknown', 'yes bank']

Saved label_maps_v3.json


In [6]:
# Split
train_df, val_df = train_test_split(
    df[['text', 'label_cat', 'label_method', 'label_bank']],
    test_size=0.15, random_state=42, stratify=df['category']
)
print(f"Train: {len(train_df)}, Val: {len(val_df)}")

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

# Tokenize
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_fn(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_fn, batched=True)
tokenized_val = val_dataset.map(tokenize_fn, batched=True)

Train: 36125, Val: 6375


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/36125 [00:00<?, ? examples/s]

Map:   0%|          | 0/6375 [00:00<?, ? examples/s]

In [7]:
class MultiHeadDistilBERT(nn.Module):
    """DistilBERT encoder with three classification heads."""

    def __init__(self, num_cat, num_method, num_bank):
        super().__init__()
        self.encoder = AutoModel.from_pretrained('distilbert-base-uncased')
        hidden = self.encoder.config.hidden_size  # 768
        self.dropout = nn.Dropout(0.1)
        self.head_cat = nn.Linear(hidden, num_cat)
        self.head_method = nn.Linear(hidden, num_method)
        self.head_bank = nn.Linear(hidden, num_bank)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids=None, attention_mask=None,
                label_cat=None, label_method=None, label_bank=None, **kwargs):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = self.dropout(outputs.last_hidden_state[:, 0])  # [CLS] token

        logits_cat = self.head_cat(cls_output)
        logits_method = self.head_method(cls_output)
        logits_bank = self.head_bank(cls_output)

        loss = None
        if label_cat is not None:
            loss_cat = self.loss_fn(logits_cat, label_cat)
            loss_method = self.loss_fn(logits_method, label_method)
            loss_bank = self.loss_fn(logits_bank, label_bank)
            loss = loss_cat + 0.5 * loss_method + 0.5 * loss_bank

        return {
            'loss': loss,
            'logits_cat': logits_cat,
            'logits_method': logits_method,
            'logits_bank': logits_bank,
        }

model = MultiHeadDistilBERT(num_cat, num_method, num_bank)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Model on {device}")
print(f"Heads: cat={num_cat}, method={num_method}, bank={num_bank}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model on cuda
Heads: cat=10, method=3, bank=20


In [8]:
from torch.utils.data import DataLoader

# Custom collate for multi-label
def collate_fn(batch):
    input_ids = torch.tensor([b['input_ids'] for b in batch])
    attention_mask = torch.tensor([b['attention_mask'] for b in batch])
    label_cat = torch.tensor([b['label_cat'] for b in batch])
    label_method = torch.tensor([b['label_method'] for b in batch])
    label_bank = torch.tensor([b['label_bank'] for b in batch])
    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'label_cat': label_cat,
        'label_method': label_method,
        'label_bank': label_bank,
    }

train_loader = DataLoader(tokenized_train, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(tokenized_val, batch_size=64, shuffle=False, collate_fn=collate_fn)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

Train batches: 1129, Val batches: 100


In [9]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR

NUM_EPOCHS = 4
LR = 2e-5

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = LinearLR(optimizer, start_factor=1.0, end_factor=0.0, total_iters=total_steps)

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        loss = out['loss']
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()
        if step % 100 == 0:
            print(f"  Epoch {epoch+1}/{NUM_EPOCHS} step {step}/{len(train_loader)} loss={loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    correct_cat = correct_method = correct_bank = total = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            pred_cat = out['logits_cat'].argmax(dim=1)
            pred_method = out['logits_method'].argmax(dim=1)
            pred_bank = out['logits_bank'].argmax(dim=1)
            correct_cat += (pred_cat == batch['label_cat']).sum().item()
            correct_method += (pred_method == batch['label_method']).sum().item()
            correct_bank += (pred_bank == batch['label_bank']).sum().item()
            total += len(batch['label_cat'])

    print(f"Epoch {epoch+1}: loss={avg_loss:.4f} | "
          f"cat_acc={correct_cat/total:.4f} | "
          f"method_acc={correct_method/total:.4f} | "
          f"bank_acc={correct_bank/total:.4f}")

  Epoch 1/4 step 0/1129 loss=4.6670
  Epoch 1/4 step 100/1129 loss=2.0235
  Epoch 1/4 step 200/1129 loss=1.4234
  Epoch 1/4 step 300/1129 loss=1.0915
  Epoch 1/4 step 400/1129 loss=1.2225
  Epoch 1/4 step 500/1129 loss=0.6650
  Epoch 1/4 step 600/1129 loss=0.8034
  Epoch 1/4 step 700/1129 loss=0.7124
  Epoch 1/4 step 800/1129 loss=0.4942
  Epoch 1/4 step 900/1129 loss=0.8984
  Epoch 1/4 step 1000/1129 loss=0.7354
  Epoch 1/4 step 1100/1129 loss=0.7566
Epoch 1: loss=1.1285 | cat_acc=0.9896 | method_acc=0.9529 | bank_acc=0.7702
  Epoch 2/4 step 0/1129 loss=0.7532
  Epoch 2/4 step 100/1129 loss=0.6029
  Epoch 2/4 step 200/1129 loss=0.4353
  Epoch 2/4 step 300/1129 loss=0.6434
  Epoch 2/4 step 400/1129 loss=0.4242
  Epoch 2/4 step 500/1129 loss=0.7760
  Epoch 2/4 step 600/1129 loss=0.6226
  Epoch 2/4 step 700/1129 loss=0.5352
  Epoch 2/4 step 800/1129 loss=0.7340
  Epoch 2/4 step 900/1129 loss=0.5283
  Epoch 2/4 step 1000/1129 loss=0.6019
  Epoch 2/4 step 1100/1129 loss=0.5755
Epoch 2: los

In [10]:
# Save the model for local inference
import os, shutil

SAVE_DIR = "my_finetuned_distilbert_v3"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save encoder + heads
model.encoder.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save head weights separately
torch.save({
    'head_cat': model.head_cat.state_dict(),
    'head_method': model.head_method.state_dict(),
    'head_bank': model.head_bank.state_dict(),
    'num_cat': num_cat,
    'num_method': num_method,
    'num_bank': num_bank,
}, os.path.join(SAVE_DIR, 'heads.pt'))

# Copy label maps
shutil.copy('label_maps_v3.json', os.path.join(SAVE_DIR, 'label_maps_v3.json'))

print(f"Saved to {SAVE_DIR}/")
print("Files:", os.listdir(SAVE_DIR))

# Zip for download
shutil.make_archive('my_model_v3', 'zip', '.', SAVE_DIR)
print("Zipped to my_model_v3.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to my_finetuned_distilbert_v3/
Files: ['label_maps_v3.json', 'config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'heads.pt']
Zipped to my_model_v3.zip


In [11]:
# Quick inference test
model.eval()
test_texts = [
    "paid 230 for electricity via slice UPI",
    "Swiggy se pizza mangwaya 450 rupaye gpay se",
    "Amazon purchase 2000 hdfc card",
    "sent 500 to Rahul on PhonePe",
    "petrol 1500 cash",
]

for text in test_texts:
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128).to(device)
    with torch.no_grad():
        out = model(**inputs)
    cat = le_cat.classes_[out['logits_cat'].argmax(dim=1).item()]
    method = le_method.classes_[out['logits_method'].argmax(dim=1).item()]
    bank = le_bank.classes_[out['logits_bank'].argmax(dim=1).item()]
    print(f"{text}")
    print(f"  -> category={cat}, method={method}, bank={bank}\n")

paid 230 for electricity via slice UPI
  -> category=utilities, method=upi, bank=unknown

Swiggy se pizza mangwaya 450 rupaye gpay se
  -> category=food, method=upi, bank=unknown

Amazon purchase 2000 hdfc card
  -> category=shopping, method=card, bank=hdfc

sent 500 to Rahul on PhonePe
  -> category=friends, method=upi, bank=unknown

petrol 1500 cash
  -> category=travel, method=cash, bank=unknown

